In [ ]:
import pandas as pd

import matplotlib.pyplot as plt

In [ ]:
chirps = pd.read_csv('../../../chirps/precip-chirps-1981-2024.csv', parse_dates=['time'])
chirps['month'] = chirps['time'].dt.month

chirps_norm = chirps[chirps['time'] <= '2010-12-31']
chirps_norm = chirps_norm.groupby(['city_residency', 'month']).quantile([0.025, 0.5, 0.975]).reset_index()

chirps_norm = chirps_norm.pivot_table(index=['month', 'city_residency'], columns='level_2', values='precip').reset_index()

chirps = chirps.set_index(['city_residency', 'month']).join(
    chirps_norm.set_index(['city_residency', 'month'])
).reset_index()

region_map = {
    'AC': 'North', 'AP': 'North', 'AM': 'North', 'PA': 'North', 'RO': 'North', 'RR': 'North', 'TO': 'North',
    'AL': 'Northeast', 'BA': 'Northeast', 'CE': 'Northeast', 'MA': 'Northeast', 'PB': 'Northeast',
    'PE': 'Northeast', 'PI': 'Northeast', 'RN': 'Northeast', 'SE': 'Northeast',
    'DF': 'Middle-West', 'GO': 'Middle-West', 'MT': 'Middle-West', 'MS': 'Middle-West',
    'ES': 'Southeast', 'MG': 'Southeast', 'RJ': 'Southeast', 'SP': 'Southeast',
    'PR': 'South', 'RS': 'South', 'SC': 'South'
}

chirps['state_residency'] = chirps['city_residency'].apply(lambda x: x.split('_')[0])
chirps['region'] = chirps['state_residency'].map(region_map)

chirps = chirps.groupby(['region', 'time'])[['precip', 0.025, 0.5, 0.975]].mean(numeric_only=True).reset_index()

chirps

In [ ]:
plt.rcParams.update({'font.size': 14})


from matplotlib.lines import Line2D
from matplotlib.patches import Patch

chirps['region'] = chirps['region'].replace({'Middle-West': 'Centre-West'})

fig, axarr = plt.subplots(2, 5, figsize=(15,8), sharey=True, sharex=True)
plt.subplots_adjust(wspace=0.2, hspace=0.3)

for ax, region in zip(axarr[0].flatten(), ['North', 'Northeast', 'Centre-West', 'Southeast', 'South']):
    chirps[(chirps['region'] == region) & (chirps['time'] >= '2023-06-01')].plot(x='time', y='precip', ax=ax, color='k', legend=False, lw=2.5)
    
    chirps[(chirps['region'] == region) & (chirps['time'] >= '2023-06-01')].plot(x='time', y=0.5, ax=ax, color='grey', lw=2, legend=False)
    ax.fill_between(
        chirps[(chirps['region'] == region) & (chirps['time'] >= '2023-06-01')]['time'].values,
        chirps[(chirps['region'] == region) & (chirps['time'] >= '2023-06-01')][0.025],
        chirps[(chirps['region'] == region) & (chirps['time'] >= '2023-06-01')][0.975],
        alpha=0.5, color='grey'
    )

    ax.set_title(region)
    ax.set_xlim(pd.to_datetime('2023-08-01'), pd.to_datetime('2024-07-01'))
    ax.set_xticks(pd.date_range('2023-08-01', '2024-07-01', freq='MS'))
    ax.set_xticklabels(['A', 'S', 'O', 'N', 'D', 'J', 'F', 'M', 'A', 'M', 'J', 'J'])
    ax.set_xlabel('Month')

legend_elements = []

legend_elements.append(Line2D([0], [0], color='k', lw=2.5, label='2023/2024'))
legend_elements.append(Line2D([0], [0], color='grey', lw=2, label='Climatology (1981-2010)'))
legend_elements.append(Patch(facecolor='grey', alpha=0.5, label='95% CI (1981-2010)'))

# Place legend on the figure level or first axis
fig.legend(handles=legend_elements, frameon=False, ncol=4, bbox_to_anchor=(0.80, 0.52))

ext_region = pd.read_csv('prec_ext_region.csv', parse_dates=['time'])
nat_region = pd.read_csv('prec_nat_region.csv', parse_dates=['time'])

ext_region['region'] = ext_region['region'].replace({'Middle-West': 'Centre-West'})
nat_region['region'] = nat_region['region'].replace({'Middle-West': 'Centre-West'})


for ax, region in zip(axarr[1].flatten(), ['North', 'Northeast', 'Centre-West', 'Southeast', 'South']):
    ext_sub = ext_region[(ext_region['region'] == region)].groupby('time').quantile([0.025, 0.5, 0.975], numeric_only=True).reset_index()
    nat_sub = nat_region[(nat_region['region'] == region)].groupby('time').quantile([0.025, 0.5, 0.975], numeric_only=True).reset_index()

    ext_sub['time'] = ext_sub['time'].apply(lambda x: pd.to_datetime(x.strftime('%Y-%m-01')))
    nat_sub['time'] = nat_sub['time'].apply(lambda x: pd.to_datetime(x.strftime('%Y-%m-01')))
    
    ext_sub[ext_sub['level_1'] == 0.5].plot(x='time', y='pr', color='C1', lw=2, ax=ax, label='ALL', legend=False)
    ax.fill_between(
        ext_sub[ext_sub['level_1'] == 0.5]['time'].values,
        ext_sub[ext_sub['level_1'] == 0.025]['pr'],
        ext_sub[ext_sub['level_1'] == 0.975]['pr'],
        alpha=0.5, color='C1'
    )

    nat_sub[nat_sub['level_1'] == 0.5].plot(x='time', y='pr', color='C0', lw=2, ax=ax, label='NAT', legend=False)
    ax.fill_between(
        nat_sub[ext_sub['level_1'] == 0.5]['time'].values,
        nat_sub[ext_sub['level_1'] == 0.025]['pr'],
        nat_sub[ext_sub['level_1'] == 0.975]['pr'],
        alpha=0.5, color='C0'
    )

    chirps[(chirps['region'] == region) & (chirps['time'] >= '2023-06-01')].plot(
        x='time', y='precip', ax=ax, color='k', lw=2.5, label='ERA5', legend=False
    )

    ax.set_xlim(pd.to_datetime('2023-08-01'), pd.to_datetime('2024-07-01'))
    ax.set_xticks(pd.date_range('2023-08-01', '2024-07-01', freq='MS'))
    ax.set_xticklabels(['A', 'S', 'O', 'N', 'D', 'J', 'F', 'M', 'A', 'M', 'J', 'J'])
    ax.set_xlabel('Month')
    #ax.legend(frameon=False)


legend_elements = []

legend_elements.append(Line2D([0], [0], color='C0', lw=2, label='NAT'))
legend_elements.append(Line2D([0], [0], color='C1', lw=2, label='ACT'))
legend_elements.append(Line2D([0], [0], color='k', lw=2.5, label='ERA5'))

# Place legend on the figure level or first axis
fig.legend(handles=legend_elements, frameon=False, ncol=4, bbox_to_anchor=(0.64, 0.05))

axarr[0, 0].set_ylabel('Precipitation (mm)')
axarr[1, 0].set_ylabel('Precipitation (mm)')

#plt.tight_layout()
plt.savefig('precip_seasonal.pdf', dpi=300, bbox_inches='tight')

In [ ]:
models = [
  'Climate(lag 1-5) + Year|region + Month|region',
]

act_all = []
nat_all = []

for model in models:
    print('-' * 60)
    print(f'Model: {model}')
    
    act = pd.read_csv(f'/gws/ssde/j25a/cpdn_nonnerc/aaim/dengue/sensitivity-fixedprec-act/predictions_regional_{model}.csv', parse_dates=['date_first_symptoms'])
    nat = pd.read_csv(f'/gws/ssde/j25a/cpdn_nonnerc/aaim/dengue/sensitivity-fixedprec-nat/predictions_regional_{model}.csv', parse_dates=['date_first_symptoms'])
    
    regions = ['North', 'Northeast', 'Middle-West', 'Southeast', 'South']
    
    act = act[act['region'].isin(regions)]
    nat = nat[nat['region'].isin(regions)]
    
    act = act.groupby([
        'model_type', 'region', 'bootstrap_iteration', 'ensemble_member'
    ]).apply(lambda x: pd.Series({
        'predicted_cases': (x['predicted_incidence'] * x['total_population'] / 1e5).sum(),
        'actual_cases': (x['actual_incidence'] * x['total_population'] / 1e5).sum(),
    }), include_groups=False).reset_index()
    
    nat = nat.groupby([
        'model_type', 'region', 'bootstrap_iteration', 'ensemble_member'
    ]).apply(lambda x: pd.Series({
        'predicted_cases': (x['predicted_incidence'] * x['total_population'] / 1e5).sum(),
        'actual_cases': (x['actual_incidence'] * x['total_population'] / 1e5).sum(),
    }), include_groups=False).reset_index()
    
    act = act.groupby(['model_type', 'region', 'bootstrap_iteration']).mean().reset_index()
    nat = nat.groupby(['model_type', 'region', 'bootstrap_iteration']).mean().reset_index()

    act_all.append(act)
    nat_all.append(nat)

act = pd.concat(act_all)
nat = pd.concat(nat_all)

diff = act.set_index(
    ['model_type', 'region', 'bootstrap_iteration']
)[['predicted_cases', 'actual_cases']].rename(
    columns={'predicted_cases': 'act', 'actual_cases': 'fitted'}
).join(
    nat.set_index(
        ['model_type', 'region', 'bootstrap_iteration']
    )[['predicted_cases']].rename(
        columns={'predicted_cases': 'nat'}  
    )
).reset_index()
diff['diff'] = 100 * (diff['act'] - diff['nat']) / diff['fitted']
diff = diff[(diff['diff'] > -200) & (diff['diff'] < 200)]
diff = diff.reset_index()
diff['region'] = diff['region'].replace({'Middle-West': 'Centre-West'})

diff_fixedprec = diff.copy()

diff_fixedprec

In [ ]:
models = [
  'Climate(lag 1-5) + Year|region + Month|region',
]

act_all = []
nat_all = []

for model in models:
    print('-' * 60)
    print(f'Model: {model}')
    
    act = pd.read_csv(f'/gws/ssde/j25a/cpdn_nonnerc/aaim/dengue/sensitivity-fixedtemp-act/predictions_regional_{model}.csv', parse_dates=['date_first_symptoms'])
    nat = pd.read_csv(f'/gws/ssde/j25a/cpdn_nonnerc/aaim/dengue/sensitivity-fixedtemp-nat/predictions_regional_{model}.csv', parse_dates=['date_first_symptoms'])
    
    regions = ['North', 'Northeast', 'Middle-West', 'Southeast', 'South']
    
    act = act[act['region'].isin(regions)]
    nat = nat[nat['region'].isin(regions)]
    
    act = act.groupby([
        'model_type', 'region', 'bootstrap_iteration', 'ensemble_member'
    ]).apply(lambda x: pd.Series({
        'predicted_cases': (x['predicted_incidence'] * x['total_population'] / 1e5).sum(),
        'actual_cases': (x['actual_incidence'] * x['total_population'] / 1e5).sum(),
    }), include_groups=False).reset_index()
    
    nat = nat.groupby([
        'model_type', 'region', 'bootstrap_iteration', 'ensemble_member'
    ]).apply(lambda x: pd.Series({
        'predicted_cases': (x['predicted_incidence'] * x['total_population'] / 1e5).sum(),
        'actual_cases': (x['actual_incidence'] * x['total_population'] / 1e5).sum(),
    }), include_groups=False).reset_index()
    
    act = act.groupby(['model_type', 'region', 'bootstrap_iteration']).mean().reset_index()
    nat = nat.groupby(['model_type', 'region', 'bootstrap_iteration']).mean().reset_index()

    act_all.append(act)
    nat_all.append(nat)

act = pd.concat(act_all)
nat = pd.concat(nat_all)

diff = act.set_index(
    ['model_type', 'region', 'bootstrap_iteration']
)[['predicted_cases', 'actual_cases']].rename(
    columns={'predicted_cases': 'act', 'actual_cases': 'fitted'}
).join(
    nat.set_index(
        ['model_type', 'region', 'bootstrap_iteration']
    )[['predicted_cases']].rename(
        columns={'predicted_cases': 'nat'}  
    )
).reset_index()
diff['diff'] = 100 * (diff['act'] - diff['nat']) / diff['fitted']
diff = diff[(diff['diff'] > -200) & (diff['diff'] < 200)]
diff = diff.reset_index()
diff['region'] = diff['region'].replace({'Middle-West': 'Centre-West'})

diff_fixedtemp = diff.copy()

diff_fixedtemp

In [ ]:
models = [
  'Climate(lag 1-5) + Year|region + Month|region',
]

act_all = []
nat_all = []

for model in models:
    print('-' * 60)
    print(f'Model: {model}')
    
    act = pd.read_csv(f'/gws/ssde/j25a/cpdn_nonnerc/aaim/dengue/sensitivity-act/predictions_regional_{model}.csv', parse_dates=['date_first_symptoms'])
    nat = pd.read_csv(f'/gws/ssde/j25a/cpdn_nonnerc/aaim/dengue/sensitivity-nat/predictions_regional_{model}.csv', parse_dates=['date_first_symptoms'])
    
    regions = ['North', 'Northeast', 'Middle-West', 'Southeast', 'South']
    
    act = act[act['region'].isin(regions)]
    nat = nat[nat['region'].isin(regions)]
    
    act = act.groupby([
        'model_type', 'region', 'bootstrap_iteration', 'ensemble_member'
    ]).apply(lambda x: pd.Series({
        'predicted_cases': (x['predicted_incidence'] * x['total_population'] / 1e5).sum(),
        'actual_cases': (x['actual_incidence'] * x['total_population'] / 1e5).sum(),
    }), include_groups=False).reset_index()
    
    nat = nat.groupby([
        'model_type', 'region', 'bootstrap_iteration', 'ensemble_member'
    ]).apply(lambda x: pd.Series({
        'predicted_cases': (x['predicted_incidence'] * x['total_population'] / 1e5).sum(),
        'actual_cases': (x['actual_incidence'] * x['total_population'] / 1e5).sum(),
    }), include_groups=False).reset_index()
    
    act = act.groupby(['model_type', 'region', 'bootstrap_iteration']).mean().reset_index()
    nat = nat.groupby(['model_type', 'region', 'bootstrap_iteration']).mean().reset_index()

    act_all.append(act)
    nat_all.append(nat)

act = pd.concat(act_all)
nat = pd.concat(nat_all)

diff = act.set_index(
    ['model_type', 'region', 'bootstrap_iteration']
)[['predicted_cases', 'actual_cases']].rename(
    columns={'predicted_cases': 'act', 'actual_cases': 'fitted'}
).join(
    nat.set_index(
        ['model_type', 'region', 'bootstrap_iteration']
    )[['predicted_cases']].rename(
        columns={'predicted_cases': 'nat'}  
    )
).reset_index()
diff['diff'] = 100 * (diff['act'] - diff['nat']) / diff['fitted']
diff = diff[(diff['diff'] > -200) & (diff['diff'] < 200)]
diff = diff.reset_index()
diff['region'] = diff['region'].replace({'Middle-West': 'Centre-West'})

diff_nofixed = diff.copy()

diff_nofixed

In [ ]:
models = [
  'Climate(lag 1-5) + Year|region + Month|region',
]

act_all = []
nat_all = []

for model in models:
    print('-' * 60)
    print(f'Model: {model}')
    
    act = pd.read_csv(f'/gws/ssde/j25a/cpdn_nonnerc/aaim/dengue/sensitivity-bspsprec-act/predictions_regional_{model}.csv', parse_dates=['date_first_symptoms'])
    nat = pd.read_csv(f'/gws/ssde/j25a/cpdn_nonnerc/aaim/dengue/sensitivity-bspsprec-nat/predictions_regional_{model}.csv', parse_dates=['date_first_symptoms'])
    
    regions = ['North', 'Northeast', 'Middle-West', 'Southeast', 'South']
    
    act = act[act['region'].isin(regions)]
    nat = nat[nat['region'].isin(regions)]
    
    act = act.groupby([
        'model_type', 'region', 'bootstrap_iteration', 'ensemble_member'
    ]).apply(lambda x: pd.Series({
        'predicted_cases': (x['predicted_incidence'] * x['total_population'] / 1e5).sum(),
        'actual_cases': (x['actual_incidence'] * x['total_population'] / 1e5).sum(),
    }), include_groups=False).reset_index()
    
    nat = nat.groupby([
        'model_type', 'region', 'bootstrap_iteration', 'ensemble_member'
    ]).apply(lambda x: pd.Series({
        'predicted_cases': (x['predicted_incidence'] * x['total_population'] / 1e5).sum(),
        'actual_cases': (x['actual_incidence'] * x['total_population'] / 1e5).sum(),
    }), include_groups=False).reset_index()
    
    act = act.groupby(['model_type', 'region', 'bootstrap_iteration']).mean().reset_index()
    nat = nat.groupby(['model_type', 'region', 'bootstrap_iteration']).mean().reset_index()

    act_all.append(act)
    nat_all.append(nat)

act = pd.concat(act_all)
nat = pd.concat(nat_all)

diff = act.set_index(
    ['model_type', 'region', 'bootstrap_iteration']
)[['predicted_cases', 'actual_cases']].rename(
    columns={'predicted_cases': 'act', 'actual_cases': 'fitted'}
).join(
    nat.set_index(
        ['model_type', 'region', 'bootstrap_iteration']
    )[['predicted_cases']].rename(
        columns={'predicted_cases': 'nat'}  
    )
).reset_index()
diff['diff'] = 100 * (diff['act'] - diff['nat']) / diff['fitted']
diff = diff[(diff['diff'] > -200) & (diff['diff'] < 200)]
diff = diff.reset_index()
diff['region'] = diff['region'].replace({'Middle-West': 'Centre-West'})

diff_bspsprec = diff.copy()

diff_bspsprec

In [ ]:
diff_fixedtemp['model_type'] = diff_fixedtemp['model_type'].replace({'Climate(lag 1-5) + Year|region + Month|region': 'Fixed Temp. / Prec (2023-24)'})
diff_fixedprec['model_type'] = diff_fixedprec['model_type'].replace({'Climate(lag 1-5) + Year|region + Month|region': 'Fixed Prec. / Temp (2023-24)'})
diff_nofixed['model_type'] = diff_nofixed['model_type'].replace({'Climate(lag 1-5) + Year|region + Month|region': 'Prec (2023-24) / Temp (2023-24)'})
diff_bspsprec['model_type'] = diff_bspsprec['model_type'].replace({'Climate(lag 1-5) + Year|region + Month|region': 'Nsplines - Prec (2023-24)'})

diff = pd.concat([diff_fixedprec, diff_fixedtemp, diff_nofixed, diff_bspsprec])

diff

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7), constrained_layout=True)

sns.boxplot(
    y='diff',
    x='region',
    hue='model_type',
    data=diff,
    ax=ax,
    gap=.2,
    order=['North', 'Northeast', 'Centre-West', 'Southeast', 'South'],
    palette={
        'Fixed Temp. / Prec (2023-24)': 'C3',
        'Fixed Prec. / Temp (2023-24)': 'C4',
        'Prec (2023-24) / Temp (2023-24)': 'C5',
        'Nsplines - Prec (2023-24)': 'C6'
    },
    showfliers=False
)

ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.15),
    frameon=False,
    fontsize=10,
    ncols=2
)
#ax.set_ylim([-100, 100])
ax.set_ylabel("Normalized Attribution Signal\n(ACT - NAT) (%)")
ax.set_xlabel("Region")
ax.axhline(0, color='k', ls='--')
plt.savefig('prec_sensitivity.pdf', dpi=300, bbox_inches='tight')